# CrackSense AI – Concrete Surface Crack Detection

This notebook builds a binary image classification model for concrete surface crack detection using transfer learning in Google Colab.

**Classes**
- Negative = No Crack
- Positive = Crack


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Import libraries

In [ ]:
import os
import zipfile
import shutil
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator


## 3. Upload crack dataset ZIP

In [ ]:
from google.colab import files
uploaded = files.upload()


## 4. Extract dataset

In [ ]:
zip_path = "/content/Crack Detection.zip"
extract_path = "/content/crack_dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)


## 5. Inspect folder structure

In [ ]:
for root, dirs, files in os.walk(extract_path):
    if files:
        print(root, "->", len(files), "files")


## 6. Find dataset root automatically

In [ ]:
possible_roots = []
for root, dirs, files in os.walk(extract_path):
    dir_names = set(dirs)
    if "Negative" in dir_names and "Positive" in dir_names:
        possible_roots.append(root)

print("Possible dataset roots:", possible_roots)

dataset_root = possible_roots[0] if possible_roots else None
print("Selected dataset root:", dataset_root)


## 7. Count images in each class

In [ ]:
negative_dir = os.path.join(dataset_root, "Negative")
positive_dir = os.path.join(dataset_root, "Positive")

negative_images = [f for f in os.listdir(negative_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
positive_images = [f for f in os.listdir(positive_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print("Negative (No Crack):", len(negative_images))
print("Positive (Crack):", len(positive_images))
print("Total:", len(negative_images) + len(positive_images))


## 8. Visualize sample images

In [ ]:
plt.figure(figsize=(12, 6))

for i, file in enumerate(negative_images[:3], 1):
    img = mpimg.imread(os.path.join(negative_dir, file))
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title("No Crack")
    plt.axis("off")

for i, file in enumerate(positive_images[:3], 4):
    img = mpimg.imread(os.path.join(positive_dir, file))
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title("Crack")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 9. Create train / validation / test split

This cell creates a new folder structure:
- train/no_crack, train/crack
- val/no_crack, val/crack
- test/no_crack, test/crack

In [ ]:
split_root = "/content/crack_split"

if os.path.exists(split_root):
    shutil.rmtree(split_root)

for split in ["train", "val", "test"]:
    for cls in ["no_crack", "crack"]:
        os.makedirs(os.path.join(split_root, split, cls), exist_ok=True)

def split_and_copy(files, src_dir, class_name, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    random.seed(seed)
    files = files.copy()
    random.shuffle(files)

    total = len(files)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_files = files[:train_end]
    val_files = files[train_end:val_end]
    test_files = files[val_end:]

    for f in train_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(split_root, "train", class_name, f))
    for f in val_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(split_root, "val", class_name, f))
    for f in test_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(split_root, "test", class_name, f))

    return len(train_files), len(val_files), len(test_files)

neg_counts = split_and_copy(negative_images, negative_dir, "no_crack")
pos_counts = split_and_copy(positive_images, positive_dir, "crack")

print("No Crack split:", neg_counts)
print("Crack split:", pos_counts)


## 10. Confirm split sizes

In [ ]:
for split in ["train", "val", "test"]:
    no_crack_count = len(os.listdir(os.path.join(split_root, split, "no_crack")))
    crack_count = len(os.listdir(os.path.join(split_root, split, "crack")))
    print(f"{split}: no_crack={no_crack_count}, crack={crack_count}, total={no_crack_count + crack_count}")


## 11. Create data generators

In [ ]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    os.path.join(split_root, "train"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary"
)

val_generator = test_datagen.flow_from_directory(
    os.path.join(split_root, "val"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(split_root, "test"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False
)

print("Class indices:", train_generator.class_indices)


## 12. Build model using MobileNetV2 transfer learning

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 13. Define callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="/content/best_crack_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


## 14. Train model

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)


## 15. Plot training history

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


## 16. Evaluate on test set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


## 17. Classification report

In [ ]:
pred_probs = model.predict(test_generator)
pred_labels = (pred_probs > 0.5).astype(int).flatten()
true_labels = test_generator.classes

target_names = ["crack", "no_crack"] if list(test_generator.class_indices.keys()) == ["crack", "no_crack"] else list(test_generator.class_indices.keys())
print("Target names:", target_names)

print(classification_report(true_labels, pred_labels, target_names=list(test_generator.class_indices.keys())))


## 18. Confusion matrix

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(test_generator.class_indices.keys())
)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()


## 19. Predict a single new image

In [ ]:
from google.colab import files
uploaded_single = files.upload()


In [ ]:
from tensorflow.keras.preprocessing import image

single_image_path = list(uploaded_single.keys())[0]

img = image.load_img(single_image_path, target_size=img_size)
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)[0][0]

if prediction > 0.5:
    predicted_class = "no_crack"
    confidence = prediction
else:
    predicted_class = "crack"
    confidence = 1 - prediction

print("Predicted class:", predicted_class)
print("Confidence:", float(confidence))


## 20. Save model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
save_dir = "/content/drive/MyDrive/CrackSenseAI"
os.makedirs(save_dir, exist_ok=True)

shutil.copy("/content/best_crack_model.keras", save_dir)
print("Saved best model to:", save_dir)


## 21. Optional fine-tuning

Unfreeze the top layers of MobileNetV2 for a few extra epochs if you want potentially higher accuracy.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=callbacks
)
